# Simplified: cross-modal closed-form (FC↔SC speed-run)

Speed-run version of `crossmodal_pca_pls_closed_form_overview.ipynb`. Strips the
exploratory scaffolding and runs only what's needed to produce the three end-of-day
tables we trust: the cross-step full-panel table, the asymmetry ratios, and the
anatomy-vs-FC identifiability triad.

## What this notebook does

Builds 8 predictions and reports 6 metrics for each, all on the same single seed-0
HCP-YA Glasser test split (n=195 subjects):

1. **Anatomy baselines** (2 predictions): `brain-vol → SC` and `brain-vol → FC` via OLS on
   16 FreeSurfer brain-volume features. Tests how much of cross-modal prediction is
   just brain anatomy.
2. **Raw cross-modal PLS** (2 predictions): `FC → SC` and `SC → FC` via the project's
   `CrossModal_PCA_PLS` architecture (PCA + PLS + inverse-PCA). Sanity check — should
   reproduce the project's ~0.134 / ~0.090 demeaned-r.
3. **Residual cross-modal PLS** (2 predictions): same as #2 but predicting the
   *anatomy-residualized* target. The headline test: does FC carry SC-specific signal
   above brain anatomy?
4. **Residual cross-modal BR** (2 predictions): same as #3 with BayesianRidge per
   PCA-latent component instead of PLS. Model-class robustness check.

## Headline finding (recalibrated 2026-05-26)

> The FC→SC > SC→FC asymmetry is robust across `demeaned_pearson`, `avg_rank`, and
> `top1_acc`. `avg_rank` is the most stable metric (~1.20-1.25× across every
> condition). Anatomy wins demeaned-r against FC (`brain-vol → SC` = 0.167 vs
> `FC → SC` = 0.132) but loses identifiability (`brain-vol top1` = 0.103 vs
> `FC top1` = 0.154). FC carries real subject-specific signal about SC orthogonal
> to anatomy (`FC → SC_residual top1` ≈ 14× chance). Pivot rationale validated.

## How to read this notebook

Top-to-bottom, ~5-10 minutes runtime on CPU. Each code cell is self-explanatory
to anyone familiar with PCA, PLS, BayesianRidge, and the HCP setup. For full
historical context (why this notebook exists, what was deferred, the bug-fix
journey, metric conventions) see `dev-notes/session_log_2026-05-26.md`,
`dev-notes/metrics_glossary.md`, and `dev-notes/notebook_architecture.md`.


In [ ]:
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import BayesianRidge, LinearRegression

import main
import models.registry
import models.eval.evaluator
import models.architectures.crossmodal_pca_pls

importlib.reload(models.architectures.crossmodal_pca_pls)
importlib.reload(models.eval.evaluator)
importlib.reload(models.registry)
importlib.reload(main)

from main import Sim
from models.eval.metrics import compute_corr_matrix, compute_basic_regression_metrics

REPO_ROOT = Path.cwd()
PARCELLATION = "Glasser"
SHUFFLE_SEED = 0
DATA_LOAD_MODE = "precomputed"

# Build Sim only to extract the raw edge arrays + indices + brain-vol features.
# The SOURCE/TARGET pair below doesn't matter for what follows -- we run our own
# pipeline on the raw edges from `base`, in both directions.
sim = Sim(
    model_name="CrossModal_PCA_PLS",
    config_path=str(REPO_ROOT / "models/configs/CrossModal_PCA_PLS.yml"),
    source="FC", target="SC",
    parcellation=PARCELLATION, shuffle_seed=SHUFFLE_SEED,
    data_load_mode=DATA_LOAD_MODE,
)
base = sim.base

# Extract clean edge-space arrays. Glasser upper triangle = 64,620 edges per connectome.
train_idx = base.trainvaltest_partition_indices["train"]
test_idx  = base.trainvaltest_partition_indices["test"]

X_anat_train = base.fs_volumes_z[train_idx]                                   # (683, 16)
X_anat_test  = base.fs_volumes_z[test_idx]                                    # (195, 16)
SC_train = np.asarray(base.sc_upper_triangles[train_idx], dtype=np.float32)   # (683, 64620)
SC_test  = np.asarray(base.sc_upper_triangles[test_idx],  dtype=np.float32)   # (195, 64620)
FC_train = np.asarray(base.fc_upper_triangles[train_idx], dtype=np.float32)
FC_test  = np.asarray(base.fc_upper_triangles[test_idx],  dtype=np.float32)

# Train means used by the demeaned_pearson metric (subtract before evaluating).
SC_train_mean = SC_train.mean(axis=0)
FC_train_mean = FC_train.mean(axis=0)

print(f"Split (family-aware, seed={SHUFFLE_SEED}): "
      f"{len(train_idx)} train / {len(test_idx)} test subjects")
print(f"Brain-vol features: {X_anat_train.shape[1]} "
      f"({', '.join(base.fs_volume_columns)})")
print(f"SC edges per subject: {SC_train.shape[1]}, FC edges per subject: {FC_train.shape[1]}")


## Helpers

Three functions, used everywhere downstream:

- **`pca_pls_predict`** — the project's `CrossModal_PCA_PLS` architecture, rewritten
  inline. PCA on source → PCA on target → PLS in latent space → inverse-PCA back to
  edges. Uses `max_iter=2000` (default 500 hits convergence warnings on this data).
- **`br_pca_predict`** — same architecture with BayesianRidge per target PCA
  component instead of PLS. Model-class swap. Slower (one BR fit per target
  component) but a different inductive bias — useful for robustness.
- **`full_panel`** — returns all 6 metrics matching the project's
  `compute_basic_regression_metrics` + `compute_demeaned_pearson_r` conventions
  exactly. The demeaned-r is computed as per-subject cosine of (y − train_mean), NOT
  row-Pearson on demeaned data — these differ by ~5% on raw predictions; for
  residual predictions they agree.


In [ ]:
def pca_pls_predict(X_src_train, X_src_test, Y_train,
                     k_src=256, k_tgt=256, k_pls=64):
    """PCA on source -> PCA on target -> PLS in latent -> inverse-PCA.

    This is exactly the project's `CrossModal_PCA_PLS` architecture (see
    `models/architectures/crossmodal_pca_pls.py`), rewritten inline so we can
    swap arbitrary targets (raw SC, residualized SC, etc.) without going
    through the project's evaluator pipeline.

    Returns Y_test_pred in the original edge space, shape (N_test, F_tgt).
    """
    pca_src = PCA(n_components=k_src, random_state=0).fit(X_src_train)
    pca_tgt = PCA(n_components=k_tgt, random_state=0).fit(Y_train)
    Z_src_train = pca_src.transform(X_src_train)
    Z_src_test  = pca_src.transform(X_src_test)
    Z_tgt_train = pca_tgt.transform(Y_train)
    pls = PLSRegression(n_components=k_pls, scale=True, max_iter=2000)
    pls.fit(Z_src_train, Z_tgt_train)
    Z_tgt_test_pred = pls.predict(Z_src_test)
    return pca_tgt.inverse_transform(Z_tgt_test_pred).astype(np.float32)


def br_pca_predict(X_src_train, X_src_test, Y_train,
                    k_src=256, k_tgt=256):
    """Same architecture as `pca_pls_predict` but BayesianRidge per target
    PCA component instead of a single PLS factorization. Different inductive
    bias (per-component shrinkage vs shared latent), useful as a model-class
    robustness check. ~1-2 min per direction for k_tgt=256 fits.
    """
    pca_src = PCA(n_components=k_src, random_state=0).fit(X_src_train)
    pca_tgt = PCA(n_components=k_tgt, random_state=0).fit(Y_train)
    Z_src_train = pca_src.transform(X_src_train)
    Z_src_test  = pca_src.transform(X_src_test)
    Z_tgt_train = pca_tgt.transform(Y_train)
    Z_tgt_test_pred = np.zeros((Z_src_test.shape[0], k_tgt), dtype=np.float32)
    for k in range(k_tgt):
        br = BayesianRidge(max_iter=300).fit(Z_src_train, Z_tgt_train[:, k])
        Z_tgt_test_pred[:, k] = br.predict(Z_src_test)
    return pca_tgt.inverse_transform(Z_tgt_test_pred).astype(np.float32)


def full_panel(y_pred, y_true, train_mean):
    """All 6 metrics matching the project's evaluator conventions exactly.

    Returns dict: mse, r2, pearson, demeaned_pearson, top1_acc, avg_rank.

    Convention notes:
      - mse, r2, pearson, top1_acc, avg_rank: from `compute_basic_regression_metrics`
        with the raw row-Pearson corr matrix. Matches the project.
      - demeaned_pearson: per-subject cosine similarity of (y - train_mean) vectors,
        averaged over subjects. Matches `compute_demeaned_pearson_r` in the project.
        For raw predictions this differs from row-Pearson-on-demeaned by ~5%; for
        residual predictions the two agree exactly (residual row means are ~0).
    """
    yp = np.asarray(y_pred,    dtype=np.float32)
    yt = np.asarray(y_true,    dtype=np.float32)
    mu = np.asarray(train_mean, dtype=np.float32)
    cc_raw = compute_corr_matrix(yt, yp)
    panel = compute_basic_regression_metrics(
        yp, yt, corr_matrix=cc_raw, corr_matrix_demeaned=None
    )
    # Project's demeaned-r convention: cosine of (y - mu) per subject, then mean.
    yp_dm = yp - mu
    yt_dm = yt - mu
    num   = (yp_dm * yt_dm).sum(axis=1)
    den_p = np.sqrt((yp_dm ** 2).sum(axis=1))
    den_t = np.sqrt((yt_dm ** 2).sum(axis=1))
    panel["demeaned_pearson"] = float((num / (den_p * den_t + 1e-10)).mean())
    return panel


print("Helpers defined: pca_pls_predict, br_pca_predict, full_panel")


## 1. Anatomy baseline + residualization

Fit OLS from 16 FreeSurfer brain-volume features (intracranial vol, gray matter,
white matter, etc.) to every connectome edge. Pure anatomy, no functional or
structural connectivity information.

**Two uses for these predictions:**

1. **Baseline numbers** — `brain-vol → SC` and `brain-vol → FC` predictions enter
   the cross-step table directly. Quantifies how much cross-modal prediction is
   "free" from anatomy alone.
2. **Residualization** — subtract the brain-vol prediction from the true target to
   get the part of SC (or FC) that anatomy *can't* explain. The downstream
   cross-modal PLS/BR runs predict these residuals → tests whether cross-modal
   signal exists *above and beyond* anatomy.


In [ ]:
# 1. Fit per-edge OLS  brain-vol -> SC  and  brain-vol -> FC.
ols_SC = LinearRegression().fit(X_anat_train, SC_train)
ols_FC = LinearRegression().fit(X_anat_train, FC_train)

# 2. Predictions on train (needed for residualization) and test (for evaluation).
SC_anat_train = ols_SC.predict(X_anat_train).astype(np.float32)
SC_anat_test  = ols_SC.predict(X_anat_test ).astype(np.float32)
FC_anat_train = ols_FC.predict(X_anat_train).astype(np.float32)
FC_anat_test  = ols_FC.predict(X_anat_test ).astype(np.float32)

# 3. Residuals: target minus anatomy prediction. Train residuals have mean ~0 by
#    construction (OLS centers); test residuals don't, but we use train mean for
#    demeaned-r so this is fine.
SC_resid_train = (SC_train - SC_anat_train).astype(np.float32)
SC_resid_test  = (SC_test  - SC_anat_test ).astype(np.float32)
FC_resid_train = (FC_train - FC_anat_train).astype(np.float32)
FC_resid_test  = (FC_test  - FC_anat_test ).astype(np.float32)

print(f"Brain-vol predictions and anatomy-residualized targets built.")
print(f"  SC residual train mean: {SC_resid_train.mean():+.2e}  (should be ~0)")
print(f"  FC residual train mean: {FC_resid_train.mean():+.2e}  (should be ~0)")
print(f"  SC residual std (vs original SC std {SC_train.std():.4f}): {SC_resid_train.std():.4f}")
print(f"  FC residual std (vs original FC std {FC_train.std():.4f}): {FC_resid_train.std():.4f}")


## 2. Raw cross-modal PLS (sanity check)

Run the manual PCA→PLS→inverse-PCA pipeline on raw SC and raw FC, both directions.
This should reproduce the project's published numbers from `CrossModal_PCA_PLS`:
- `FC → SC` demeaned ~ **0.132–0.134** (project's 10-seed mean: 0.1341 ± 0.0054)
- `SC → FC` demeaned ~ **0.085–0.090** (project's 10-seed mean: 0.0904 ± 0.0101)

If these don't roughly match, the manual pipeline isn't calibrated and we can't
trust the residual results downstream.


In [ ]:
# Raw FC -> SC via PCA + PLS + inverse-PCA.
FC_to_SC_raw_pred = pca_pls_predict(FC_train, FC_test, SC_train)
# Raw SC -> FC (mirror).
SC_to_FC_raw_pred = pca_pls_predict(SC_train, SC_test, FC_train)
print("Raw cross-modal PLS predictions built.")
print("Expected demeaned-r:")
print("  FC -> SC raw ~ 0.132  (project 10-seed: 0.1341 +/- 0.0054)")
print("  SC -> FC raw ~ 0.085  (project 10-seed: 0.0904 +/- 0.0101)")


## 3. Residual cross-modal PLS — the headline test

Same architecture as #2, but predicting `SC_residual` and `FC_residual` instead of
raw SC and FC. The question this answers:

> *Once we strip out the part of SC/FC that brain anatomy can explain, how much
> cross-modal signal remains?*

This is the test that validates (or invalidates) the FC→SC pivot. Specifically:
- If `FC → SC_residual` demeaned-r is near zero, then FC→SC's apparent advantage
  was just FC capturing brain size.
- If it's clearly nonzero AND larger than `SC → FC_residual`, FC carries real
  subject-specific information about SC beyond anatomy, and the asymmetry survives
  anatomy control.

**Spoiler:** the second case is what we see. `FC → SC_resid` ≈ 0.078, `SC → FC_resid`
≈ 0.056, ratio ≈ 1.4×.


In [ ]:
# FC -> SC_residual: does FC predict the part of SC anatomy can't explain?
FC_to_SC_resid_pls_pred = pca_pls_predict(FC_train, FC_test, SC_resid_train)
# SC -> FC_residual: the mirror, for the asymmetry comparison.
SC_to_FC_resid_pls_pred = pca_pls_predict(SC_train, SC_test, FC_resid_train)
print("Residual cross-modal PLS predictions built.")


## 4. Residual cross-modal BR (model-class robustness)

Same residual prediction task as #3, but with BayesianRidge per target PCA
component instead of a single PLS factorization. Different inductive bias:
- PLS finds a small number of shared source-target latent directions and projects
  jointly.
- BR fits independent ridge regressions per target component with empirical Bayes
  shrinkage.

If the FC→SC > SC→FC asymmetry is real, it should appear under both. If it shows
up only under PLS, the asymmetry might be a quirk of the PLS objective.

Cost: 256 BR fits per direction (~1-2 min total).


In [ ]:
print("Fitting FC -> SC_residual (BR, 256 per-component fits)...")
FC_to_SC_resid_br_pred = br_pca_predict(FC_train, FC_test, SC_resid_train)
print("Fitting SC -> FC_residual (BR, 256 per-component fits)...")
SC_to_FC_resid_br_pred = br_pca_predict(SC_train, SC_test, FC_resid_train)
print("Residual cross-modal BR predictions built.")


## 5. Final tables

Evaluate all 8 predictions through `full_panel` (all 6 metrics) and assemble:

1. **Cross-step full-panel table** — every prediction × every metric, 8 rows × 6
   metric columns.
2. **Asymmetry ratios table** — for each model class (raw / resid PLS / resid BR),
   compute `FC→SC / SC→FC` ratio per metric. Tests whether the directional
   asymmetry is consistent across metrics.
3. **Anatomy vs FC identifiability triad** — head-to-head on `demeaned_pearson`,
   `top1_acc`, `avg_rank`. Where the methodologically interesting finding lives
   (anatomy wins demeaned-r, FC wins identifiability).


In [ ]:
# Each row: (display name, group label, prediction array, truth array, train_mean
# vector for demeaning). Order is: 2 anatomy, 2 raw PLS, 2 residual PLS, 2 residual BR.
rows = [
    ("brain-vol -> SC",        "anatomy",         SC_anat_test,            SC_test,       SC_train_mean),
    ("brain-vol -> FC",        "anatomy",         FC_anat_test,            FC_test,       FC_train_mean),
    ("FC -> SC raw (PLS)",     "raw FC->SC",      FC_to_SC_raw_pred,        SC_test,       SC_train_mean),
    ("SC -> FC raw (PLS)",     "raw SC->FC",      SC_to_FC_raw_pred,        FC_test,       FC_train_mean),
    ("FC -> SC_resid (PLS)",   "resid FC->SC PLS", FC_to_SC_resid_pls_pred, SC_resid_test, SC_resid_train.mean(axis=0)),
    ("SC -> FC_resid (PLS)",   "resid SC->FC PLS", SC_to_FC_resid_pls_pred, FC_resid_test, FC_resid_train.mean(axis=0)),
    ("FC -> SC_resid (BR)",    "resid FC->SC BR",  FC_to_SC_resid_br_pred,  SC_resid_test, SC_resid_train.mean(axis=0)),
    ("SC -> FC_resid (BR)",    "resid SC->FC BR",  SC_to_FC_resid_br_pred,  FC_resid_test, FC_resid_train.mean(axis=0)),
]
panels = {name: full_panel(pred, true, mu) for name, _, pred, true, mu in rows}

# ---- Table 1: cross-step full-panel table ----
df = pd.DataFrame([
    {"experiment": name, "group": grp, **panels[name]}
    for name, grp, _, _, _ in rows
])[["experiment", "group", "mse", "r2", "pearson",
    "demeaned_pearson", "top1_acc", "avg_rank"]]
print("=== CROSS-STEP FULL-PANEL TABLE ===\n")
print(df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# ---- Table 2: asymmetry ratios (FC->SC / SC->FC) across metrics ----
def ratio_row(label, fc_sc, sc_fc):
    return {
        "comparison": label,
        "demeaned_pearson": fc_sc["demeaned_pearson"] / max(sc_fc["demeaned_pearson"], 1e-12),
        "top1_acc":         fc_sc["top1_acc"]         / max(sc_fc["top1_acc"],         1e-12),
        "avg_rank":         fc_sc["avg_rank"]         / max(sc_fc["avg_rank"],         1e-12),
        "pearson":          fc_sc["pearson"]          / max(sc_fc["pearson"],          1e-12),
    }

ratios = pd.DataFrame([
    ratio_row("raw     PLS   FC->SC / SC->FC", panels["FC -> SC raw (PLS)"],     panels["SC -> FC raw (PLS)"]),
    ratio_row("residual PLS  FC->SC / SC->FC", panels["FC -> SC_resid (PLS)"],   panels["SC -> FC_resid (PLS)"]),
    ratio_row("residual BR   FC->SC / SC->FC", panels["FC -> SC_resid (BR)"],    panels["SC -> FC_resid (BR)"]),
])
print("\n\n=== ASYMMETRY RATIOS (FC->SC / SC->FC) ACROSS METRICS ===\n")
print(ratios.to_string(index=False, float_format=lambda x: f"{x:.3f}x"))

# ---- Table 3: anatomy vs FC head-to-head on the identifiability triad ----
id_tri = pd.DataFrame([
    {"predictor": "brain-vol  -> SC", **{k: panels["brain-vol -> SC"][k]       for k in ("demeaned_pearson","top1_acc","avg_rank")}},
    {"predictor": "FC raw     -> SC", **{k: panels["FC -> SC raw (PLS)"][k]    for k in ("demeaned_pearson","top1_acc","avg_rank")}},
    {"predictor": "FC residual-> SC", **{k: panels["FC -> SC_resid (PLS)"][k]  for k in ("demeaned_pearson","top1_acc","avg_rank")}},
    {"predictor": "brain-vol  -> FC", **{k: panels["brain-vol -> FC"][k]       for k in ("demeaned_pearson","top1_acc","avg_rank")}},
    {"predictor": "SC raw     -> FC", **{k: panels["SC -> FC raw (PLS)"][k]    for k in ("demeaned_pearson","top1_acc","avg_rank")}},
    {"predictor": "SC residual-> FC", **{k: panels["SC -> FC_resid (PLS)"][k]  for k in ("demeaned_pearson","top1_acc","avg_rank")}},
])
print("\n\n=== ANATOMY vs FC ON IDENTIFIABILITY TRIAD ===\n")
print(id_tri.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


## How to read the three tables

### Table 1 — cross-step full panel

Each row is one prediction. Six metrics tell layered stories:

- `mse`, `r2` — total reconstruction error / variance explained. Often negative
  r² because we're predicting onto a 64,620-dim target with N=683 train. **Ignore
  small differences.**
- `pearson` — raw per-subject Pearson, dominated by the group connectome pattern.
  ~0.83–0.91 for raw predictions; collapses to the same value as `demeaned_pearson`
  for residual predictions (residual row means are ~0). **Sanity check, not a
  scientific signal.**
- `demeaned_pearson` — per-subject cosine of (y − train_mean). **Headline metric**
  for individual subject signal. ~0.13 for raw FC→SC, ~0.08 for residual.
- `top1_acc` — strict identifiability (correct subject as #1 match in 195-gallery).
  Chance = 1/195 ≈ 0.005. Discrete and noisy at this sample size.
- `avg_rank` — average rank percentile of the correct match. **Most stable
  metric** in our results; ratios sit at 1.20–1.25× across every condition.

### Table 2 — asymmetry ratios

For each model class, the FC→SC / SC→FC ratio per metric. If all four metrics
show a ratio > 1, the directional asymmetry is real and metric-robust.

Expected pattern (also what we observe):
- All ratios > 1 ✓
- `avg_rank` ratio is the most stable across model classes (1.20–1.25×)
- `top1_acc` ratio is largest but noisiest (1.8–5.3×)
- `demeaned_pearson` ratio is the headline (~1.4×)

### Table 3 — anatomy vs FC identifiability

The methodologically interesting finding: **metrics disagree on which predictor is
"better."** Brain-vol wins `demeaned_pearson` for SC target (0.167 vs 0.132), but
FC wins `top1_acc` (0.154 vs 0.103). On FC target, anatomy gets `top1_acc = 0.000`
— zero correct identifications out of 195.

**Reading hint:** if you only look at `demeaned_pearson`, anatomy looks like it
dominates and "FC→SC is just predicting brain size." If you also look at `top1_acc`
and `avg_rank`, FC carries identifying information that anatomy cannot.

## Headline (recalibrated 2026-05-26)

1. **FC→SC > SC→FC asymmetry is robust** across all three identifiability metrics,
   in every model class × framework combination tested.
2. **Anatomy is the dominant confounder of the raw asymmetry**, but the residual
   asymmetry survives anatomy control (~1.4×).
3. **Anatomy ≠ "the real signal"** — wins demeaned-r against FC, loses on
   identifiability. SC and FC are anatomy-driven to very different degrees
   (`brain-vol → SC` demeaned 0.167 vs `brain-vol → FC` 0.047; SC is 3.6× more
   anatomy-driven).
4. **FC carries real subject-specific signal about SC orthogonal to anatomy**
   (`FC → SC_residual top1 ≈ 14× chance`). Pivot rationale validated.

## What's not here (lives elsewhere)

- 10-seed CIs on FC→SC raw (Step 2 in the original notebook): demeaned 0.1341 ± 0.0054.
- Bootstrap CIs on top1/avg_rank: pending — see `notebooks-FC_to_SC-experimental/model_overviews/todo.md`.
- SC→SC self-prediction oracle (ceiling): demeaned 0.647, top1 1.0, avg_rank 1.0.
- Variance partition framework (Bayesian, K_TGT-sensitive): walked back. See
  `dev-notes/phase0_snapshots.md` for the history.
- The methodology bug-fix journey for the demeaned-r formula and PLS convergence:
  see `dev-notes/metrics_glossary.md`.

For full context: `dev-notes/session_log_2026-05-26.md`.
